# Notebook 02: Physics RLVR — GRPO Training

Reinforcement learning from verifiable rewards (RLVR) on top of the SFT warmup model.
The model learns to produce well-structured reasoning chains that lead to the correct
MMLU-Pro physics answer.

This notebook is the second stage in a two-stage fine-tuning pipeline:
1. **Notebook 01 (SFT warmup)**: supervises the model to follow the XML output schema
2. **This notebook (GRPO)**: uses RL to reward correct answers, pushing accuracy higher

**Input**: `physics_rlvr/data/grpo_train.jsonl` — 1,000 physics MCQ examples  
**Input**: `physics_rlvr/models/qwen3-4b-physics-sft-lora/` — LoRA adapters from SFT warmup  
**Output**: `physics_rlvr/models/qwen3-4b-physics-grpo/` — merged 16-bit weights for eval

---

## Reward Design

| Reward | Range | Purpose |
|--------|-------|---------|
| Format | 0 – 0.02 | XML schema compliance |
| Reasoning | −0.1 – 0.10 | Shaped reasoning length |
| Correctness | 0.0 – 1.0 | Letter match (dominant signal) |

**Correctness is binary (1.0 or 0.0)** — MCQ has no partial credit.
GRPO's group normalization provides the gradient signal via group advantage:
`advantage_i = (r_i - mean(r)) / std(r)` over G rollouts per prompt.

Groups where all G completions are correct or all wrong have std=0 → **zero gradient**.
Only mixed-outcome groups fire. Correctness is the signal that matters.

## Step 1: Mount Google Drive and Install Dependencies

Three library versions are pinned to match the working configuration from the base coding project:

- **TRL 0.22.2**: In this version `completions` inside reward functions are **raw strings**. Newer TRL passes dicts — writing `completion[0]["content"]` would silently break all three reward signals with no error.
- **unsloth-zoo 2026.1.2**: A regression introduced in Feb 2026 broke the `masked_batch_mean` function inside the compiled `GRPOTrainer`. Pinning to the Jan 2026 build avoids it.
- **transformers 4.56.2**: Locks tokenizer and model-loading APIs to the versions unsloth 2026.1.2 expects.

`UNSLOTH_VLLM_STANDBY=1` keeps the vLLM inference server alive between training batches, eliminating repeated cold-start overhead on every rollout step.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
%cd /content/drive/MyDrive/grpo-verified-reasoner
!ls

Mounted at /content/drive
/content/drive/MyDrive/grpo-verified-reasoner
data			      models	    README.md			 wandb
grpo_trainer_lora_model       notebooks     src
huggingface_tokenizers_cache  outputs	    unsloth_compiled_cache
LICENSE			      physics_rlvr  _unsloth_sentencepiece_temp


In [ ]:
import shutil
for p in [
    "/tmp/unsloth_compiled_cache",
    "/content/unsloth_compiled_cache",
    "/content/drive/MyDrive/grpo-verified-reasoner/unsloth_compiled_cache",
]:
    shutil.rmtree(p, ignore_errors=True)
print("Cleared Unsloth compiled caches.")


Cleared Unsloth compiled caches.


In [ ]:
!nvidia-smi

Fri Feb 27 07:23:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   28C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# Install UV (faster pip)
!pip install --upgrade -qqq uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 120.9 MB/s eta 0:00:00


In [ ]:
import os
import subprocess

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:False"
os.environ["UNSLOTH_VLLM_STANDBY"]    = "1"   # Same as old project
os.environ["WANDB_PROJECT"]            = "physics-rlvr"

In [ ]:
# Environment Logic (Colab vs Local) — copied verbatim from notebooks/05_GRPO_optimization_2.ipynb
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    # Version Matching
    try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
    except: get_numpy = "numpy"; get_pil = "pillow"
    try: is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False

    # A100/H100/G4 gets vllm 0.10.2, T4 gets 0.9.2 (same as old project)
    get_vllm, get_triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.10.2", "triton")

    # Install Everything
    !uv pip install -qqq --upgrade \
        unsloth {get_vllm} {get_numpy} {get_pil} torchvision bitsandbytes xformers
    !uv pip install -qqq {get_triton}

# Pin TRL and transformers to versions used in the existing coding project
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

# Upgrade wandb to fix protobuf mismatch in Colab
!uv pip install -q --upgrade wandb

# Pin unsloth-zoo to the exact version from the old project (Jan 5, 2026)
# Current (Feb 2026) version has a masked_batch_mean bug in the compiled GRPOTrainer
!uv pip install --no-deps --force-reinstall unsloth==2026.1.2 unsloth-zoo==2026.1.2


Using Python 3.12.12 environment at: /usr
Resolved 18 packages in 47ms
Prepared 1 package in 233ms
Uninstalled 1 package in 30ms
Installed 1 package in 30ms
 - transformers==4.57.6
 + transformers==4.56.2
Using Python 3.12.12 environment at: /usr
Resolved 1 package in 0.84ms
Prepared 1 package in 89ms
Uninstalled 1 package in 0.98ms
Installed 1 package in 5ms
 - trl==0.24.0
 + trl==0.22.2
Using Python 3.12.12 environment at: /usr
Resolved 2 packages in 43ms
Prepared 2 packages in 89ms
Uninstalled 2 packages in 1ms
Installed 2 packages in 4ms
 - unsloth==2026.2.1
 + unsloth==2026.1.2
 - unsloth-zoo==2026.2.1
 + unsloth-zoo==2026.1.2


In [ ]:
# Upgrade wandb to fix protobuf mismatch in Colab
!uv pip install -q --upgrade wandb

In [ ]:
import re
import torch
import wandb
import numpy as np
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer
from vllm import SamplingParams

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 02-27 07:24:51 [__init__.py:216] Automatically detected platform cuda.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not import trl.trainer.alignprop_trainer: Failed to import trl.trainer.alignprop_trainer because of the following error (look up to see its traceback):
Failed to import trl.models.modeling_sd_base because of the following error (look up to see its traceback):
Failed to import diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion because of the following error (look up to see its traceback):
Failed to import diffusers.loaders.single_file because of the following error (look up to see its traceback):
name 'logger' is not defined
Unsloth: Could not import trl.trainer.ddpo_trainer: Failed to import trl.trainer.ddpo_trainer because of the following error (look up to see its traceback):
Failed to import trl.models.modeling_sd_base because of the following er

In [ ]:
wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: samyakshrestha (samyakshrestha-university-of-texas-at-dallas) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
import importlib.metadata as md
for p in ["unsloth","unsloth-zoo","vllm","trl","transformers"]:
    print(p, md.version(p))

unsloth 2026.1.2
unsloth-zoo 2026.1.2
vllm 0.10.2
trl 0.22.2
transformers 4.56.2


## Step 2: Verify GPU and Environment

GPU type determines `num_generations` (G) in Step 9:
- **H100 (80 GB)**: G=16 — more rollouts per prompt → lower-variance advantage estimates → stronger learning signal
- **A100 / other**: G=8 — conservative default to avoid OOM

The RTX PRO 6000 Blackwell (97 GB VRAM) is detected as "other" in the GPU name check, so this run uses G=8.

In [ ]:
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.8.0+cu128
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


## Step 3: Load SFT Model with LoRA Adapters

Load the SFT warmup model from the saved LoRA adapter path.
`fast_inference=True` enables vLLM-backed rollout generation during GRPO training.
No `get_peft_model` call needed — the LoRA adapters are already embedded in the checkpoint.

In [ ]:
MODEL_PATH     = "physics_rlvr/models/qwen3-4b-physics-sft-lora"
MAX_SEQ_LENGTH = 3072

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name             = MODEL_PATH,
    max_seq_length         = MAX_SEQ_LENGTH,
    load_in_4bit           = False,    # Full precision for RL stability
    fast_inference         = True,     # Required for vLLM-backed rollouts
    gpu_memory_utilization = 0.8,
)

INFO 02-27 07:25:54 [vllm_utils.py:702] Unsloth: Patching vLLM v1 graph capture
INFO 02-27 07:25:54 [vllm_utils.py:731] Unsloth: Patching vLLM v0 graph capture
==((====))==  Unsloth 2026.1.2: Fast Qwen3 patching. Transformers: 4.56.2. vLLM: 0.10.2.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Standby mode is enabled. Changing `gpu_memory_utilization` to 0.925.
Unsloth: vLLM loading unsloth/Qwen3-4B-Base with actual GPU utilization = 91.88%
Unsloth: Your GPU has CUDA compute capability 12.0 with VRAM = 94.97 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 3072. Num Sequences = 256.
Unsloth: vLLM's KV 

`torch_dtype` is deprecated! Use `dtype` instead!


INFO 02-27 07:26:07 [__init__.py:1815] Using max model len 3072
INFO 02-27 07:26:07 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 02-27 07:26:07 [lora.py:92] `lora_extra_vocab_size` is deprecated and will be removed in v0.12.0. Additional vocabulary support for LoRA adapters is being phased out.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

INFO 02-27 07:28:04 [gpu_worker.py:391] Free memory on device (94.25/94.97 GiB) on startup. Desired GPU memory utilization is (0.9188347243992774, 87.26 GiB). Actual usage is 7.81 GiB for weight, 1.43 GiB for peak activation, 0.02 GiB for non-torch memory, and 0.66 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=82897167360` to fit into requested memory, or `--kv-cache-memory=90399876096` to fully utilize gpu memory. Current kv cache memory in use is 83765388288 bytes.
INFO 02-27 07:28:04 [core.py:218] init engine (profile, create kv cache, warmup model) took 93.69 seconds
INFO 02-27 07:28:04 [llm.py:295] Supported_tasks: ('generate',)
INFO 02-27 07:28:04 [__init__.py:36] No IOProcessor plugins requested by the model
Unsloth: Standby mode is enabled. Pre-sleeping vLLM model to reduce OOMs.
Unsloth: Just some info: will skip parsing ['post_feedforward_layernorm', 'post_layernorm', 'norm', 'k_norm', 'input_layernorm', 'ffn_norm', 'post_attention_la

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of Qwen3ForCausalLM were not initialized from the model checkpoint at unsloth/Qwen3-4B-Base and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['post_feedforward_layernorm', 'post_layernorm', 'cross_attn_input_layernorm', 'norm', 'k_norm', 'input_layernorm', 'ffn_norm', 'post_attention_layernorm', 'norm1', 'layer_norm1', 'cross_attn_post_attention_layernorm', 'pre_feedforward_layernorm', 'norm2', 'attention_norm', 'q_norm', 'layer_norm2']


Unsloth 2026.1.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [ ]:
# Verify LoRA adapters loaded and trainable parameters exist
trainable = 0
total     = 0
for name, p in model.named_parameters():
    n      = p.numel()
    total += n
    if p.requires_grad:
        trainable += n

print(f"Trainable params: {trainable:,} / {total:,} = {100*trainable/total:.4f}%")
assert trainable > 0, "No trainable parameters found — LoRA adapters may not have loaded correctly."
print("LoRA adapters confirmed. Ready for GRPO training.")

Trainable params: 66,060,288 / 4,088,528,384 = 1.6157%
LoRA adapters confirmed. Ready for GRPO training.


## Step 4: Pre-Training Sanity Check

Verify the SFT model produces well-formed XML output before starting the multi-hour GRPO run.

In [ ]:
# Identical to SFT notebook — must match exactly
SYSTEM_PROMPT = """You are a physics reasoning engine.

You must output your response in the following exact format:

<START_WORKING_OUT>
Step-by-step reasoning to solve the physics problem.
</END_WORKING_OUT>
<SOLUTION>
Single letter answer only (A, B, C, D, E, F, G, H, I, or J).
</SOLUTION>

Do not output anything outside these tags."""

In [ ]:
test_q    = "A car moving at 20 m/s brakes uniformly and stops in 4 seconds. What is its deceleration?"
test_opts = ["2 m/s\u00b2", "4 m/s\u00b2", "5 m/s\u00b2", "8 m/s\u00b2", "10 m/s\u00b2"]

option_str   = "\n".join(f"{chr(65+i)}. {opt}" for i, opt in enumerate(test_opts))
user_content = f"{test_q}\n\nOptions:\n{option_str}"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": user_content},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
)
inputs    = {k: v.to("cuda") for k, v in inputs.items()}
input_len = inputs["input_ids"].shape[1]

# Do NOT call FastLanguageModel.for_inference here — model.generate() works directly
# and we avoid any risk of inference-mode patching interfering with GRPOTrainer.
with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens = 300,
        temperature    = 0.0,
        do_sample      = False,
    )

generated_text = tokenizer.decode(output[0][input_len:], skip_special_tokens=True)
print("=== SFT MODEL OUTPUT (before GRPO) ===")
print(generated_text)

=== SFT MODEL OUTPUT (before GRPO) ===
<START_WORKING_OUT>
A: Let's think step by step. We refer to Wikipedia articles on classical mechanics for help. The formula for acceleration is a = (v_f - v_i) / t, where v_f is the final velocity, v_i is the initial velocity, and t is the time. Plugging in the values we are given, we get a = (0 - 20) / 4 = -5. The answer is (C).
</END_WORKING_OUT>
<SOLUTION>
C
</SOLUTION>


## Step 5: Define Output Schema and Extraction Helpers

`validate_schema()` checks for all four required XML tags in the correct order.
Returns `(bool, str)` — always unpack as `is_valid, _ = validate_schema(text)`.

Two extractors with different strictness:
- **Strict**: used for format reward (bare letter only)
- **Relaxed**: used for correctness reward and eval (handles `(E)`, `E.`, etc.)

In [ ]:
# Compiled regex patterns for tag validation
RE_START   = re.compile(r"<START_WORKING_OUT>",  re.IGNORECASE)
RE_END     = re.compile(r"</END_WORKING_OUT>",   re.IGNORECASE)
RE_SOL     = re.compile(r"<SOLUTION>",           re.IGNORECASE)
RE_SOL_END = re.compile(r"</SOLUTION>",          re.IGNORECASE)


def validate_schema(text: str) -> tuple[bool, str]:
    """
    Check whether the model output follows the required XML schema.
    Returns (is_valid, reason) — always unpack: is_valid, _ = validate_schema(text)
    """
    if not RE_START.search(text):   return False, "Missing <START_WORKING_OUT>"
    if not RE_END.search(text):     return False, "Missing </END_WORKING_OUT>"
    if not RE_SOL.search(text):     return False, "Missing <SOLUTION>"
    if not RE_SOL_END.search(text): return False, "Missing </SOLUTION>"
    # Tag order: reasoning block must precede solution block
    if RE_SOL.search(text).start() < RE_START.search(text).start():
        return False, "Tag order incorrect"
    return True, "Schema valid"

In [ ]:
VALID_LETTERS = set("ABCDEFGHIJ")


def extract_solution_strict(response: str):
    """
    Strict extractor — used for format reward.
    Returns a single letter A-J only if SOLUTION block contains exactly one letter.
    Returns None otherwise.
    """
    match = re.search(
        r"<SOLUTION>\s*([A-Ja-j])\s*</SOLUTION>",
        response, re.DOTALL | re.IGNORECASE
    )
    if not match:
        return None
    letter = match.group(1).strip().upper()
    return letter if letter in VALID_LETTERS else None


def extract_solution(response: str):
    """
    Relaxed extractor — used for correctness reward and evaluation.
    Finds the first valid A-J letter anywhere inside <SOLUTION>...</SOLUTION>.
    Handles early-training variants: '(E)', 'E.', 'The answer is E'.
    Returns None if no SOLUTION block or no valid letter found.
    """
    sol_match = re.search(
        r"<SOLUTION>(.*?)</SOLUTION>",
        response, re.DOTALL | re.IGNORECASE
    )
    if not sol_match:
        return None
    content      = sol_match.group(1)
    letter_match = re.search(r"[A-Ja-j]", content)
    if not letter_match:
        return None
    return letter_match.group(0).upper()

In [ ]:
# Test validators on the pre-training sanity check output
is_valid, reason = validate_schema(generated_text)
extracted        = extract_solution(generated_text)

print(f"Schema valid:     {is_valid} — {reason}")
print(f"Extracted answer: {extracted}")
print(f"\nRaw output preview (first 400 chars):\n{generated_text[:400]}")

Schema valid:     True — Schema valid
Extracted answer: C

Raw output preview (first 400 chars):
<START_WORKING_OUT>
A: Let's think step by step. We refer to Wikipedia articles on classical mechanics for help. The formula for acceleration is a = (v_f - v_i) / t, where v_f is the final velocity, v_i is the initial velocity, and t is the time. Plugging in the values we are given, we get a = (0 - 20) / 4 = -5. The answer is (C).
</END_WORKING_OUT>
<SOLUTION>
C
</SOLUTION>


## Step 6: Define Reward Functions

Three reward functions combined additively.

**Critical (TRL 0.22.2)**: `completions` are **raw strings**, not dicts.  
Do NOT write `completion[0]["content"]` — write `completion` directly.

In [ ]:
def format_reward_func(completions, **kwargs) -> list[float]:
    """
    Reward for XML schema compliance.
    Returns +0.02 for valid schema, 0.0 otherwise.
    completions: list of raw strings (TRL 0.22.2 — NOT dicts).
    """
    rewards = []
    for completion in completions:
        is_valid, _ = validate_schema(completion)
        rewards.append(0.02 if is_valid else 0.0)
    return rewards

In [ ]:
def reasoning_reward_func(completions, **kwargs) -> list[float]:
    """
    Shaped reward encouraging concise, useful reasoning.
    - < 50 chars:    0.0   (no reasoning)
    - 0–400 chars:   linear ascent 0.0 → 0.1
    - 400–800 chars: plateau at 0.1
    - > 800 chars:   penalty −0.03 per 100 chars over 800, floored at −0.1
    """
    rewards = []
    for completion in completions:
        match = re.search(
            r"<START_WORKING_OUT>(.*?)</END_WORKING_OUT>",
            completion, re.DOTALL | re.IGNORECASE
        )
        if not match:
            rewards.append(0.0)
            continue

        length = len(match.group(1).strip())
        if length < 50:
            rewards.append(0.0)
        elif length <= 400:
            rewards.append((length / 400.0) * 0.1)
        elif length <= 800:
            rewards.append(0.1)
        else:
            overage = length - 800
            score   = 0.1 - (overage / 100.0) * 0.03
            rewards.append(max(-0.1, score))
    return rewards

In [ ]:
def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    """
    Binary correctness reward: 1.0 if extracted letter matches ground truth, else 0.0.
    MCQ has no partial credit — GRPO's group advantage is the gradient signal:
      advantage_i = (r_i - mean(r_1..r_G)) / std(r_1..r_G)
    """
    rewards = []
    for completion, gt_letter in zip(completions, answer):
        predicted = extract_solution(completion)
        rewards.append(1.0 if predicted == gt_letter.upper() else 0.0)
    return rewards

## Step 7: Reward Function Smoke Test

Run all three reward functions on handcrafted completions **before** launching training.
This catches silent bugs that would only surface after hours of compute.

In [ ]:

# Handcrafted completions for smoke testing
SMOKE_CORRECT = """<START_WORKING_OUT>
Using kinematics: deceleration = change_in_velocity / time = (20 - 0) / 4 = 5 m/s squared.
The deceleration is 5 m/s^2, which corresponds to option C.
</END_WORKING_OUT>
<SOLUTION>
C
</SOLUTION>"""

SMOKE_WRONG = """<START_WORKING_OUT>
I think the answer involves momentum conservation.
The velocity changes from 20 to 0, so the change is 20 m/s.
</END_WORKING_OUT>
<SOLUTION>
B
</SOLUTION>"""

SMOKE_MALFORMED = "I think the answer is C because Newton's second law applies here."

batch_completions = [SMOKE_CORRECT, SMOKE_WRONG, SMOKE_MALFORMED]
batch_prompts     = ["dummy_prompt"] * 3
batch_answers     = ["C", "C", "C"]

r_format  = format_reward_func(completions=batch_completions)
r_reason  = reasoning_reward_func(completions=batch_completions)
r_correct = correctness_reward_func(
    prompts=batch_prompts, completions=batch_completions, answer=batch_answers
)

print("Format rewards:     ", r_format)
print("Reasoning rewards:  ", [round(r, 4) for r in r_reason])
print("Correctness rewards:", r_correct)

# Sanity assertions
assert r_format[0]  > 0,    "SMOKE_CORRECT should pass schema check"
assert r_format[2] == 0.0,  "SMOKE_MALFORMED should fail schema check"
assert r_correct[0] == 1.0, "Correct answer C should get 1.0"
assert r_correct[1] == 0.0, "Wrong answer B should get 0.0"
assert r_correct[2] == 0.0, "Malformed (no tags) should get 0.0"

print("\nAll smoke tests passed. Reward functions are correctly wired.")

Format rewards:      [0.02, 0.02, 0.0]
Reasoning rewards:   [0.0375, 0.0275, 0.0]
Correctness rewards: [1.0, 0.0, 0.0]

All smoke tests passed. Reward functions are correctly wired.


## Step 8: Load GRPO Dataset and Apply Chat Template

Load 1,000 MMLU-Pro physics training examples.
Build `prompt` (ChatML string with generation marker) + `answer` (ground truth letter) columns.
GRPOTrainer passes `answer` as a keyword argument to each reward function.

In [ ]:
DATA_PATH   = "physics_rlvr/data/grpo_train.jsonl"
raw_dataset = load_dataset("json", data_files=DATA_PATH, split="train")

print(f"Loaded {len(raw_dataset)} GRPO training examples.")
print("Columns:", raw_dataset.column_names)

ex = raw_dataset[0]
print(f"\nSample row:")
print(f"  question: {ex['question'][:80]}...")
print(f"  options:  {ex['options'][:3]} ...")
print(f"  answer:   {ex['answer']}")

assert len(raw_dataset) == 1000, f"Expected 1000 GRPO examples, got {len(raw_dataset)}"
assert "question" in raw_dataset.column_names
assert "options"  in raw_dataset.column_names
assert "answer"   in raw_dataset.column_names

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 1000 GRPO training examples.
Columns: ['question', 'options', 'answer']

Sample row:
  question: (a) What is the acceleration of gravity on the moon's surface, g_m (as compared ...
  options:  ['g_m = (1/6)g_e, 12 ft', 'g_m = (1/7)g_e, 16 ft', 'g_m = (1/4)g_e, 15 ft'] ...
  answer:   A


In [ ]:
def apply_chat_template(row):
    """
    Wrap each GRPO example in ChatML format.
    GRPOTrainer expects:
      - 'prompt': ChatML string ending with <|im_start|>assistant\n
      - 'answer': ground truth letter (passed as 'answer' kwarg to reward functions)
    """
    question   = row["question"]
    options    = row["options"]
    option_str = "\n".join(f"{chr(65+i)}. {opt}" for i, opt in enumerate(options))
    user_content = f"{question}\n\nOptions:\n{option_str}"

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]

    # tokenize=False → raw ChatML string
    # add_generation_prompt=True → appends <|im_start|>assistant\n so model generates from here
    row["prompt"] = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    return row


dataset = raw_dataset.map(apply_chat_template)

print("After applying chat template:")
print("Columns:", dataset.column_names)
print(f"\nFirst prompt (last 300 chars — should end with <|im_start|>assistant):")
print(dataset[0]["prompt"][-300:])
print(f"\nFirst answer: {dataset[0]['answer']}")

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

After applying chat template:
Columns: ['question', 'options', 'answer', 'prompt']

First prompt (last 300 chars — should end with <|im_start|>assistant):
n the moon?

Options:
A. g_m = (1/6)g_e, 12 ft
B. g_m = (1/7)g_e, 16 ft
C. g_m = (1/4)g_e, 15 ft
D. g_m = (1/3)g_e, 6 ft
E. g_m = (1/9)g_e, 18 ft
F. g_m = (1/5)g_e, 9 ft
G. g_m = (1/2)g_e, 8 ft
H. g_m = (1/8)g_e, 14 ft
I. g_m = (2/3)g_e, 3 ft
J. g_m = (1/4)g_e, 10 ft<|im_end|>
<|im_start|>assistant


First answer: A


## Step 9: Configure GRPO Hyperparameters

Key choices and rationale:

**`num_generations = G = 8`** (the central GRPO parameter)  
Each training prompt gets G independent rollouts. Per-rollout advantage is computed within the group:
`advantage_i = (r_i − mean(r_1…r_G)) / std(r_1…r_G)`  
A group where all G completions are correct or all wrong has std=0 → zero gradient (zero-advantage collapse). G=8 is the safe minimum for binary MCQ rewards; G=16 halves this risk but requires double the VRAM.

**`learning_rate = 5e-6`**  
100× lower than SFT (2e-4). RL training is sensitive to large policy updates — overshooting causes rapid policy collapse.

**`weight_decay = 0.1` + `cosine` LR + `warmup_ratio = 0.1`**  
L2 regularization and LR warm-up prevent reward hacking and policy divergence during early training.

**`gradient_accumulation_steps = 4`**  
Effective batch = 8 rollouts × 4 = 32 rollout-pairs per gradient update. A larger effective batch smooths the noisy GRPO gradient.

**`max_completion_length = 1024`**  
Hard cap paired with the reasoning reward: rewards plateau at 400–800 chars and penalize beyond 800. The 1024-token cap enforces the ceiling.

**`num_train_epochs = 1`**  
One pass over 1,000 prompts ≈ 250 optimizer steps. A single epoch is standard for GRPO on small datasets — multiple passes risk reward hacking on the training distribution.

In [ ]:
# Auto-detect GPU to set num_generations
# H100 (80GB) can fit 16 generations; A100 (40GB) uses 8 to avoid OOM
try:
    is_h100 = "H100" in torch.cuda.get_device_name(0)
except Exception:
    is_h100 = False

NUM_GENERATIONS = 16 if is_h100 else 8
print(f"GPU: {'H100' if is_h100 else 'A100/other'} → num_generations = {NUM_GENERATIONS}")

GPU: A100/other → num_generations = 8


In [ ]:
# vLLM sampling parameters for GRPO rollouts
# These govern how the model generates completions during training
vllm_sampling_params = SamplingParams(
    min_p                    = 0.1,            # Baseline diversity floor
    top_p                    = 0.95,           # Nucleus sampling cutoff
    top_k                    = -1,             # Disable top-k (use top-p only)
    seed                     = 3407,           # Reproducibility across rollouts
    temperature              = 0.8,            # Moderate diversity for GRPO
    stop                     = [tokenizer.eos_token],
    include_stop_str_in_output = True,
)

In [ ]:
training_args = GRPOConfig(
    # Experiment tracking
    output_dir = "outputs",
    report_to  = "wandb",
    run_name   = "physics-rlvr-grpo",

    # vLLM rollout configuration
    vllm_sampling_params = vllm_sampling_params,
    use_vllm             = True,

    # Optimizer
    learning_rate     = 5e-6,
    weight_decay      = 0.1,
    warmup_ratio      = 0.1,
    lr_scheduler_type = "cosine",
    optim             = "adamw_8bit",

    # Batch / generation sizing
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    num_generations             = NUM_GENERATIONS,

    # Sequence length caps
    max_prompt_length     = 1024,
    max_completion_length = 1024,

    # Training duration: 1 epoch over 1,000 examples ≈ 250 steps
    num_train_epochs = 1,

    # Logging and checkpointing
    logging_steps = 1,
    save_steps    = 50,
)

Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 8


## Step 10: Initialize and Run GRPO Trainer

`GRPOTrainer` implements the policy gradient loop:

1. **Rollout**: vLLM generates G=8 completions per prompt at temperature=0.8
2. **Score**: all three reward functions are called; outputs are summed per rollout
3. **Normalize**: group advantage is computed — `(r_i − mean) / std` within the G rollouts
4. **Update**: PPO-clip gradient step updates the LoRA parameters; frozen base weights are untouched

The three reward functions run in order: `format_reward_func` (0–0.02) → `reasoning_reward_func` (−0.1–0.10) → `correctness_reward_func` (0.0–1.0). Correctness dominates — format and reasoning rewards provide auxiliary shaping signal to discourage degenerate outputs (empty responses, runaway length).

With 1,000 prompts × 1 epoch and effective batch size 32, training runs for 250 steps (~54 minutes on this GPU).

In [ ]:
reward_functions = [
    format_reward_func,       # XML schema compliance (0.0 – 0.02)
    reasoning_reward_func,    # Shaped reasoning length (−0.1 – 0.10)
    correctness_reward_func,  # Letter match (0.0 or 1.0) — dominant signal
]

trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = reward_functions,
    args             = training_args,
    train_dataset    = dataset,
)

In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 250
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / format_reward_func / mean,rewards / format_reward_func / std,rewards / reasoning_reward_func / mean,rewards / reasoning_reward_func / std,rewards / correctness_reward_func / mean,rewards / correctness_reward_func / std
1,0.001600,0.363703,0.262729,264.468750,74.000000,414.000000,0.000000,264.468750,74.000000,414.000000,1.622307,0.020000,0.000000,0.062453,0.046285,0.281250,0.456803
2,0.000500,0.156091,0.153281,210.281250,64.000000,552.000000,0.000000,210.281250,64.000000,552.000000,0.520347,0.020000,0.000000,0.073591,0.048713,0.062500,0.245935
3,0.000500,0.744820,0.310160,182.343750,71.000000,610.000000,0.000000,182.343750,71.000000,610.000000,0.525251,0.020000,0.000000,0.068570,0.043110,0.656250,0.482559
4,0.000400,0.353278,0.399701,235.812500,108.000000,409.000000,0.000000,235.812500,108.000000,409.000000,0.393116,0.020000,0.000000,0.083278,0.033390,0.250000,0.439941
5,0.000500,0.352525,0.369673,245.000000,46.000000,1024.000000,0.031250,219.870956,46.000000,699.000000,0.485358,0.019375,0.003536,0.051900,0.060010,0.281250,0.456803
6,0.000500,0.413650,0.130150,176.468750,65.000000,444.000000,0.000000,176.468750,65.000000,444.000000,0.510353,0.020000,0.000000,0.081150,0.028821,0.312500,0.470929
7,0.001400,0.570466,0.266190,181.531250,108.000000,590.000000,0.000000,181.531250,108.000000,590.000000,1.448199,0.019375,0.003536,0.082341,0.038922,0.468750,0.507007
8,0.000400,0.880372,0.317627,175.531250,66.000000,424.000000,0.000000,175.531250,66.000000,424.000000,0.399893,0.020000,0.000000,0.079122,0.020998,0.781250,0.420013
9,0.000400,0.486077,0.479049,176.468750,84.000000,369.000000,0.000000,176.468750,84.000000,369.000000,0.389691,0.020000,0.000000,0.091077,0.017165,0.375000,0.491869
10,0.000500,0.302786,0.333378,212.968750,60.000000,638.000000,0.000000,212.968750,60.000000,638.000000,0.524466,0.020000,0.000000,0.064036,0.045559,0.218750,0.420013


profiling/Time taken: UnslothGRPOTrainer._calculate_rewards,▂▂▁▁▂▂▁▂▅▃▅▃▂▃▄▂▂▄▃▂▃▁▂▂▂▂▂▄▂▅▄█▄▃▂▃▃▃▃▃
profiling/Time taken: UnslothGRPOTrainer._prepare_inputs,▁▁▁▇▁▁▁▁▁▁▁▁▁▁▁▁▇▁▁▁▁▁▇▆▁▁▁▁▁▁▁▁█▁▁▁▇▁▁▁
profiling/Time taken: UnslothGRPOTrainer.correctness_reward_func,▂▆▂█▇▂▃▂▂▅▂▁▃▂▅▃▃▃▂▃▄▂▂▂▄▂▂▄▂▃▃▃▅▃▃▃▂▂▂▂
profiling/Time taken: UnslothGRPOTrainer.format_reward_func,▃▂▃█▃▄▂▂▁▁▄▃▂▄▂▄▂▃▃▂▃▅▄▃▇▄▃▂▃▃▅▃▅▂▃▄▄▂▄▆
profiling/Time taken: UnslothGRPOTrainer.reasoning_reward_func,▅▃▅▄▃▂▃▄▅▃▁▄▇▅▃▄▇▅▄▄▅▆▄▄▄▄▆▄█▄▆▆▄▄▂▇▃▅▃▅
profiling/Time taken: UnslothGRPOTrainer.vLLM.generate,▄▄▄█▁▃▃▁▃███▂▆▂▂▂▁▃██▅▃▅▃█▃▄▅▁▅▁▃▄█▇▄▃▄▄
train/completion_length,▆▇▃▄▆▇▃▄▅▆▁▄█▃▃▂▆▃▅▆▅▂▇▅▃▇▆▄█▇▃▇▆▆▆▆█▅▇▇
train/completions/clipped_ratio,▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁
train/completions/max_length,██▂▅▃▂▃▇▁▂▂▇▅▂▆▂▂▃▄▄▇▄▄▄▂▃▂▃▄▆▃▃█▃▃▄▄▆█▃
train/completions/max_terminated_length,▅▃▃▂▄▃▃▅▂▄▃▁▄▇▃▅▃▄▅▂▃▄▅▁▃█▇▃▄▃▇▄▄▃▃▅▃▅▃▃
+20,...


TrainOutput(global_step=250, training_loss=0.0006166510879993439, metrics={'train_runtime': 3238.2217, 'train_samples_per_second': 0.309, 'train_steps_per_second': 0.077, 'total_flos': 0.0, 'train_loss': 0.0006166510879993439})

## Step 11: Post-Training Sanity Check

Call `FastLanguageModel.for_inference(model)` to re-enable inference-mode kernel patches (disabled during RL training). Then generate a sample completion and verify the trained model still produces valid XML and a plausible answer.

This is a quick sanity check that RL training did not corrupt the model's format-following behavior — a known failure mode when reward signals are misconfigured or the KL penalty is too weak.

In [ ]:
# Switch back to inference mode after training
FastLanguageModel.for_inference(model)

test_q2    = "An object of mass 5 kg is acted upon by a net force of 20 N. What is its acceleration?"
test_opts2 = ["1 m/s\u00b2", "2 m/s\u00b2", "4 m/s\u00b2", "5 m/s\u00b2", "10 m/s\u00b2"]

option_str2   = "\n".join(f"{chr(65+i)}. {opt}" for i, opt in enumerate(test_opts2))
user_content2 = f"{test_q2}\n\nOptions:\n{option_str2}"

messages2 = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": user_content2},
]

inputs2 = tokenizer.apply_chat_template(
    messages2,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
)
inputs2    = {k: v.to("cuda") for k, v in inputs2.items()}
input_len2 = inputs2["input_ids"].shape[1]

with torch.no_grad():
    output2 = model.generate(
        **inputs2,
        max_new_tokens = 400,
        temperature    = 0.0,
        do_sample      = False,
    )

generated_text2 = tokenizer.decode(output2[0][input_len2:], skip_special_tokens=True)

print("=== GRPO MODEL OUTPUT (after training) ===")
print(generated_text2)

is_valid2, reason2 = validate_schema(generated_text2)
predicted2         = extract_solution(generated_text2)
print(f"\nSchema valid:     {is_valid2} — {reason2}")
print(f"Predicted answer: {predicted2}  (expected: C for 4 m/s\u00b2)")

=== GRPO MODEL OUTPUT (after training) ===
<START_WORKING_OUT>
A: Let's think step by step. We refer to Wikipedia articles on classical mechanics for help. Let's solve this problem step by step. We know that force equals mass times acceleration, or F = ma. We are given the mass m = 5 kg and the force F = 20 N. We can solve for the acceleration a = F/m = (20 N) / (5 kg) = 4 m/s². The answer is (C).
</END_WORKING_OUT>
<SOLUTION>
C
</SOLUTION>

Schema valid:     True — Schema valid
Predicted answer: C  (expected: C for 4 m/s²)


## Step 12: Export Training Artifacts

Export the full training log from `trainer.state.log_history` to CSV and generate reward curve plots.

The key signal to inspect is `rewards/correctness_reward_func/mean`: an upward trend confirms the model is learning to pick correct answers; a flat or declining curve signals zero-advantage collapse or reward hacking.

Plots saved to `physics_rlvr/artifacts/grpo_run1/`:
- `reward_total.png` — summed reward per step (format + reasoning + correctness)
- `reward_correctness_mean.png` — correctness mean per step (primary quality signal)
- `kl.png` — KL divergence from the reference policy (a spike signals policy collapse)
- `completion_mean_length.png` — average completion length (tracks reasoning verbosity over training)

In [ ]:
import os, json
import pandas as pd
import matplotlib.pyplot as plt

ART_DIR = "physics_rlvr/artifacts/grpo_run1"
os.makedirs(ART_DIR, exist_ok=True)

# 1) Export raw training logs
df = pd.DataFrame(trainer.state.log_history)
if "step" in df.columns:
    df = df[df["step"].notna()].copy()
df.to_csv(f"{ART_DIR}/train_log.csv", index=False)

def plot_metric(col, title, fname, smooth=10):
    if col not in df.columns:
        return
    x = df["step"] if "step" in df.columns else range(len(df))
    y = pd.to_numeric(df[col], errors="coerce")
    y_s = y.rolling(smooth, min_periods=1).mean()

    plt.figure(figsize=(8,4))
    plt.plot(x, y, alpha=0.35, label="raw")
    plt.plot(x, y_s, linewidth=2, label=f"rolling-{smooth}")
    plt.title(title)
    plt.xlabel("Step")
    plt.ylabel(col)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{ART_DIR}/{fname}", dpi=150)
    plt.close()

# 2) Key plots for slides
plot_metric("reward", "GRPO Total Reward", "reward_total.png")
plot_metric("rewards/correctness_reward_func/mean", "Correctness Reward (Mean)", "reward_correctness_mean.png")
plot_metric("rewards/reasoning_reward_func/mean", "Reasoning Reward (Mean)", "reward_reasoning_mean.png")
plot_metric("rewards/format_reward_func/mean", "Format Reward (Mean)", "reward_format_mean.png")
plot_metric("kl", "KL", "kl.png")
plot_metric("completions/clipped_ratio", "Completion Clipped Ratio", "clipped_ratio.png")
plot_metric("completions/mean_length", "Completion Mean Length", "completion_mean_length.png")

# 3) Small summary JSON
summary = {
    "final_step": int(df["step"].max()) if "step" in df.columns and len(df) else None,
    "final_reward": float(df["reward"].dropna().iloc[-1]) if "reward" in df.columns and df["reward"].notna().any() else None,
    "final_correctness_mean": float(df["rewards/correctness_reward_func/mean"].dropna().iloc[-1]) if "rewards/correctness_reward_func/mean" in df.columns and df["rewards/correctness_reward_func/mean"].notna().any() else None,
    "max_correctness_mean": float(df["rewards/correctness_reward_func/mean"].max()) if "rewards/correctness_reward_func/mean" in df.columns else None,
}
with open(f"{ART_DIR}/summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Saved artifacts to:", ART_DIR)
print("Files:", os.listdir(ART_DIR))

Saved artifacts to: physics_rlvr/artifacts/grpo_run1
Files: ['train_log.csv', 'reward_total.png', 'reward_correctness_mean.png', 'reward_reasoning_mean.png', 'reward_format_mean.png', 'kl.png', 'clipped_ratio.png', 'completion_mean_length.png', 'summary.json']


## Step 13: Save Model

Save **merged 16-bit weights** for direct vLLM loading in Notebook 03 (evaluation).  
Also save LoRA adapters as a compact backup.

- `qwen3-4b-physics-grpo-lora/` — LoRA adapter files only (~130 MB, small backup)
- `qwen3-4b-physics-grpo/` — merged 16-bit weights (required for vLLM in Notebook 03)

Note: `merged_32bit` and `merged_16bit` both produce bf16 tensors with Qwen3. Prefer the `-16bit` variant — it completes the save with `config.json` included, which is required for vLLM to load the model.

In [ ]:
# LoRA adapter backup (small, always safe)
LORA_OUT = "physics_rlvr/models/qwen3-4b-physics-grpo-lora"
os.makedirs(LORA_OUT, exist_ok=True)

model.save_lora(LORA_OUT)
tokenizer.save_pretrained(LORA_OUT)
print(f"LoRA adapters saved to: {LORA_OUT}")

LoRA adapters saved to: physics_rlvr/models/qwen3-4b-physics-grpo-lora


In [ ]:
# Merged 32-bit weights (precision-safe export for evaluation/archival)
MODEL_OUT = "physics_rlvr/models/qwen3-4b-physics-grpo"
os.makedirs(MODEL_OUT, exist_ok=True)

model.save_pretrained_merged(MODEL_OUT, tokenizer, save_method="merged_32bit")
print(f"Merged model saved to:   {MODEL_OUT}")
print("Files saved:", os.listdir(MODEL_OUT))

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...


Unsloth: Copying 2 files from cache to `physics_rlvr/models/qwen3-4b-physics-grpo`: 100%|██████████| 2/2 [00:22<00:00, 11.28s/it]


Successfully copied all 2 files from cache to `physics_rlvr/models/qwen3-4b-physics-grpo`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:44<00:00, 22.40s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/grpo-verified-reasoner/physics_rlvr/models/qwen3-4b-physics-grpo`
Merged model saved to:   physics_rlvr/models/qwen3-4b-physics-grpo
Files saved: ['chat_template.jinja', 'tokenizer_config.json', 'special_tokens_map.json', 'added_tokens.json', 'vocab.json', 'merges.txt', 'tokenizer.json', '.cache', 'model.safetensors.index.json', 'model-00001-of-00002.safetensors', 'model-00002-of-00002.safetensors']


In [ ]:
MODEL_OUT = "physics_rlvr/models/qwen3-4b-physics-grpo-16bit"
os.makedirs(MODEL_OUT, exist_ok=True)

model.save_pretrained_merged(MODEL_OUT, tokenizer, save_method="merged_16bit")
print(f"Merged model saved to:   {MODEL_OUT}")
print("Files saved:", os.listdir(MODEL_OUT))

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...


Unsloth: Copying 2 files from cache to `physics_rlvr/models/qwen3-4b-physics-grpo-16bit`: 100%|██████████| 2/2 [00:21<00:00, 10.63s/it]


Successfully copied all 2 files from cache to `physics_rlvr/models/qwen3-4b-physics-grpo-16bit`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:51<00:00, 25.91s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/grpo-verified-reasoner/physics_rlvr/models/qwen3-4b-physics-grpo-16bit`
Merged model saved to:   physics_rlvr/models/qwen3-4b-physics-grpo-16bit
Files saved: ['chat_template.jinja', 'tokenizer_config.json', 'special_tokens_map.json', 'added_tokens.json', 'vocab.json', 'merges.txt', 'tokenizer.json', 'config.json', '.cache', 'model.safetensors.index.json', 'model-00001-of-00002.safetensors', 'model-00002-of-00002.safetensors']


# Step 14: Fixing Metadata

In [1]:
import nbformat
import os

In [2]:
from google.colab import drive, files
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
# List the notebook directory to confirm the file exists
os.listdir("/content/drive/MyDrive/grpo-verified-reasoner/physics_rlvr/new")

['04_merge_checkpoint_250.ipynb',
 '03_evaluation.ipynb',
 '00_data_preparation.ipynb',
 '01_sft_warmup.ipynb',
 '02_grpo_training.ipynb']

In [ ]:
notebook_path = "/content/drive/MyDrive/grpo-verified-reasoner/physics_rlvr/new/02_grpo_training.ipynb"

with open(notebook_path, "r") as f:
    nb = nbformat.read(f, as_version=4)

if "widgets" in nb.metadata:
    del nb.metadata["widgets"]

with open(notebook_path, "w") as f:
    nbformat.write(nb, f)

print("Notebook fixed and saved successfully!")